- DPO 的核心机制就是通过最大化“正样本（Positive/Winning Sample）”与“负样本（Negative/Losing Sample）”之间的似然差来优化策略。
- 将 DPO 置于监督微调（SFT）之后，强化学习（RL）之前 。
    - 代表工作
        - https://arxiv.org/abs/2601.04888
        - MiroThinker: Pushing the Performance Boundaries of Open-Source Research Agents via Model, Context, and Interactive Scaling
    - 目标
        - 决策细化（Refinement）： SFT 阶段让模型建立了基本的 Agent 行为，而 DPO 的目的是进一步细化模型的决策能力 。
        - 目标对齐（Alignment）： 通过 DPO，使模型的决策制定与任务目标（即正确解决问题）更加一致 。
        - 概率调整： 鼓励模型提高生成“优选轨迹”（Preferred Trajectory）的概率，同时降低“非优选轨迹”的概率，从而巩固正确的推理模式 。

### reward model 的准确性

> When a measure becomes a target, it ceases to be a good measure.

$$
\max_{\pi_\theta} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta(y|x)} [r_\phi(x, y)] - \beta \mathbb{D}_{\text{KL}} [\pi_\theta(y \mid x) \parallel \pi_{\text{ref}}(y \mid x)],
$$

- The added constraint is important, as it prevents the model from deviating too far from the distribution on which the reward model is accurate, as well as maintaining the generation diversity and preventing mode-collapse to single high-reward answers.”
- 什么是“Reward Model 准确”的分布？（In-Distribution）
    - 奖励模型是准确的，当且仅当语言模型（Policy）生成的文本分布，接近于奖励模型训练数据的分布。
    - 奖励模型通常是在 SFT（有监督微调）模型生成的回复上训练的。人类标注员会对 SFT 模型生成的 $(y_1, y_2)$ 对进行打分或排序。
    - 准确区（Trust Region）： 当语言模型生成的句子在语法、逻辑、长度和风格上与 SFT 模型的输出相似时，奖励模型处于“舒适区”。此时，RM 给出的分数能真实反映人类的偏好（例如：流利、有用、无害）。
- 什么是“Reward Model 不准确”的分布？（Out-of-Distribution / OOD）
    - 当语言模型为了“刷分”而过度优化，生成了偏离人类自然语言习惯、或者利用了 RM 神经网络漏洞的文本时，RM 就会变得极不准确。

### DPO -> RM

> language model can be interpreted as a reward model
- 给定偏好对 $(y^+, y^-)$, DPO 直接更新要部署的模型 $\pi_\theta$, 让 preferred response 的相对概率变大：
    - $\beta \log \frac{\pi_\theta(y^+|x)}{\pi_{\text{ref}}(y^+|x)}>\beta \log \frac{\pi_\theta(y^-|x)}{\pi_{\text{ref}}(y^-|x)}$
- 训练结束后，$\pi_\theta$ 本身就是生成回答的 policy。
- DPO 的数学结构里天然包含一个“隐式 reward”：
    - $r_\theta(x,y)=\beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$
- 它一边改变 policy，一边让模型的 log-prob ratio 具有 reward 的含义。换句话说，DPO 的 loss 看起来在调概率，实际也在学习“偏好分数”。